# X10D2 — Project One: Group Challenge

## ExoticMeatMarkets.com — Full Dallas Expansion

Over the past several weeks you've built a complete delivery optimization pipeline:
- Geocoding addresses and building road network graphs (X9D1)
- Greedy allocation by margin and TSP routing (X9D2)
- Refactoring into reusable functions and comparing truck scenarios (X10D1)

**Today the training wheels come off.**

ExoticMeatMarkets.com has received orders from **29 Dallas restaurants** — up from 10.
The supplier has also expanded: **5 trucks** are now available to rent,
each with a different cost structure and capacity.
You may also deploy **up to two trucks simultaneously** on a given day.

### Company Headquarters — Dallas Love Field
All routes begin and end at the ExoticMeatMarkets.com distribution hub:

> **8333 George Coker Cir, Dallas, TX** *(Dallas Love Field)*

The Cuy arrives here each morning. Your truck(s) depart from this address,
complete the delivery route, and return. **Route distance must include the
leg from HQ to the first stop and the return leg from the last stop back to HQ.**

### Real-World Variability
On actual delivery days, supply levels and restaurant demand shift constantly.
**Your solution must be dynamic** — it should work correctly for any combination
of supply availability and bid prices, not just the values in `restaurants_full.xlsx`.
Your group will be given a specific supply scenario at the start of X11D2;
your notebook should be ready to re-run with new inputs in minutes.

Your group's goal: **build a solution that dynamically identifies the most profitable
1- or 2-truck configuration given any supply level, demand, and bid prices.**

---

## Project Deliverables

| Item | Due |
|------|-----|
| Group work session (in class) | Today, X10D2 — last 30 minutes |
| Full work day | X11D1 (Tuesday) |
| **Final notebook submitted to GitHub + Canvas** | **X11D2 (Thursday), start of class** |

**One submission per group.** Submit a link to your GitHub repository on Canvas.

---

## The Data Files

You have two new files for this project:

### `restaurants_presentation.xlsx`
Presentation-day restaurant data across Dallas with four columns:
- `RESTAURANT` — name
- `ADDRESS` — street address (Dallas, TX)
- `ENTREES_REQUESTED` — how many entrees each restaurant ordered
- `ENTREE_BID_PRICE` — what that restaurant is willing to pay per entree

**Bid prices range from $35 to $95.** Your supply cost for the presentation scenario is **$52.00 per entree**.
Margins are tighter than your X9 set — restaurant selection and routing decisions matter more.

⚠️ **These values represent one possible day.** On delivery day (X11D2) your instructor
will announce the actual supply available and may adjust bid prices. Your notebook
must accept `DAILY_SUPPLY_CAP` as a variable — never hardcode it.

### `trucks.csv`
Five trucks available to rent with four columns:
- `truck_name`
- `fixed_cost` — flat daily fee regardless of distance
- `per_mile_cost` — variable cost per mile driven
- `capacity` — maximum entrees the truck can carry

**No single truck dominates.** A truck with low fixed cost may have high per-mile cost.
A high-capacity truck costs more to deploy. You must run scenarios to find the winner.


## Constraints and Rules

### Starting Point — Company HQ
Every route starts **and ends** at:
> **8333 George Coker Cir, Dallas, TX** (Dallas Love Field)

Geocode this address exactly once. Snap it to its nearest OSMnx node.
The first leg of every route is HQ → first restaurant stop.
The last leg is final restaurant stop → HQ.
Total miles must include both of these legs.

### Supply Cap — Variable by Day
ExoticMeatMarkets.com can supply up to **427 entrees** in the presentation scenario,
across both trucks combined if you deploy two. **However, the actual supply
available on delivery day will be announced by your instructor at the start of X11D2
and may differ.** Your notebook must treat `DAILY_SUPPLY_CAP` as a variable
that can be changed in a single cell without breaking anything downstream.

### Single-Truck Rules
- You pick **one truck** from `trucks.csv`
- That truck's `capacity` caps how many entrees it can physically carry
- Effective cap = `min(DAILY_SUPPLY_CAP, truck.capacity)`
- Route the truck through your selected restaurants (greedy TSP, as before)
- Compute the full P&L

### Two-Truck Rules (optional — but potentially more profitable)
- You pick **two different trucks** from `trucks.csv`
- You **split the 29 restaurants** between the two trucks however you like
- Each truck gets its own allocation and its own route
- Each truck pays its own fixed + variable costs
- **Combined entrees across both trucks ≤ 427** (the daily supply cap still applies)
- **Each truck must serve at least 3 restaurants** (a one-stop delivery doesn't justify the truck)
- Net profit = Truck 1 net profit + Truck 2 net profit

### General Rules
- A restaurant cannot be served by both trucks
- You cannot serve more entrees than a restaurant requested
- Partial fills are allowed (serve fewer entrees than requested if supply runs out)
- All routes use real Dallas road distances (OSMnx graph from X9D1/X10D1)


## Scoring Rubric

| Category | Points | Description |
|----------|--------|-------------|
| **Correctness** | 25 | P&L math is right; HQ-anchored routes; constraints respected; assertions pass |
| **Code Quality** | 20 | Functions used appropriately; DRY (don't repeat your code); readable; docstrings |
| **Recommendation** | 15 | Produces clear recommendations with numbers to back it up |
| **Validation** | 10 | At least 5 assert-based checks covering key constraints and math |
| **Creativity** | 30 | A creative approach that distinguishes your solution from others (didn't just run it through Claude) |
| **Total** | 100 | |

### Correctness details
Your notebook must produce a **final P&L table** showing:
- Revenue, supply cost, gross profit
- Truck fixed cost, truck variable cost (miles × per_mile_cost)
- Net profit

If deploying two trucks, show the P&L for each truck and a combined summary row.
Miles must include the HQ departure and return legs.

### Scenario Exploration details
Your notebook must show **at least three distinct scenarios** you evaluated:
- At minimum: each single-truck option you seriously considered
- At least one two-truck scenario
- At least two different supply cap values tested (e.g., 100, 427)
- A comparison table at the end showing net profit for each scenario

### Recommendation
A markdown cell at the end of your notebook titled **"Our Recommendation"** that:
1. States which truck(s) you recommend and why
2. Cites specific numbers (net profit, miles driven, key restaurants served/skipped)
3. Acknowledges any trade-offs or assumptions in your approach
4. Explains how your recommendation would change at different supply levels

### Creativity details
There is no single formula here — this category rewards groups that go beyond the
scaffold and bring something original. Examples of what earns creativity points:
- A novel restaurant-split strategy for two trucks with a clear rationale
- A function that automatically selects the best 1-vs-2-truck configuration
  given any supply level (no manual testing required)
- A visualization that makes the trade-offs immediately obvious
- A sensitivity analysis showing at which supply level the optimal truck changes
- Any other approach that makes your solution more useful on a real delivery day

The best solutions will be ones your instructor could actually hand to the delivery
driver on the morning of X11D2 and get a confident answer in seconds.


## Hints and Suggested Approach

### Start from X10D1
Your `allocate_supply()`, `build_route()`, and `calculate_pnl()` functions from X10D1
are already the right building blocks. Copy them into your project notebook.

The main changes you'll need:
1. Load `restaurants_presentation.xlsx` instead of `restaurants_x9.xlsx`
2. Load truck configs from `trucks_presentation.csv` instead of hardcoded dicts
3. Add HQ as a fixed start/end node to `build_route()`
4. Make `DAILY_SUPPLY_CAP` easy to change — set it once at the top of your notebook
5. Loop over truck options to compare scenarios; consider automating the selection

### Geocoding HQ
Add the HQ address to your geocoding step as a special row, or geocode it separately.
Snap it to its OSMnx node just like a restaurant. Then modify `build_route()` so
it always starts from the HQ node and returns to it at the end.

```python
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'
# Geocode once and store the node:
# hq_lat, hq_lon = geocode_address(HQ_ADDRESS)
# HQ_NODE = ox.nearest_nodes(G, hq_lon, hq_lat)
```

### Dynamic supply cap
Set this once at the top of your notebook and never reference a literal number again:
```python
DAILY_SUPPLY_CAP = 427   # ← change this one line on delivery day
```
Every downstream function call should pass `DAILY_SUPPLY_CAP` as a variable.
On X11D2 your instructor may announce a different number — you should be able
to update it and re-run the entire notebook in under a minute.

### Loading trucks from CSV
```python
import pandas as pd

trucks_df = pd.read_csv('trucks.csv')
print(trucks_df)

# Convert each row to a dict — same structure as X10D1's TRUCK_A / TRUCK_B
truck_list = trucks_df.to_dict(orient='records')
# truck_list[0] looks like:
# {'truck_name': 'Sparrow', 'fixed_cost': 75.0, 'per_mile_cost': 2.5, 'capacity': 60}
```

### Auto-selecting the best configuration (Creativity opportunity)
Rather than manually inspecting a comparison table, consider writing a function
that accepts `DAILY_SUPPLY_CAP` and returns the best truck (or truck pair)
automatically:
```python
def find_best_configuration(df, truck_list, supply_cap, G, HQ_NODE):
    """
    Evaluate all single-truck scenarios and a selection of two-truck scenarios.
    Return the configuration with the highest net profit.
    """
    # TODO: your group designs this
    pass
```

### Two-truck split strategies to consider
Think about what makes a good split — these are just starting points:
- Geographic split (north Dallas vs south Dallas by latitude)
- Price tier split (high bid-price restaurants on one truck, casual on the other)
- Margin split (top N restaurants by margin on Truck 1, rest on Truck 2)
- Distance-from-HQ split (close restaurants on a small cheap truck, far ones on a larger one)

There is no one right answer. Part of the challenge is figuring out what to test.

### Watch out for
- **Geocoding:** `restaurants_presentation.xlsx` has the restaurant addresses plus HQ. Geocoding all of them from scratch
  will take ~60 seconds. Use the same cache-to-disk pattern from X9D1/X10D1.
  Name your cache file `restaurants_full_geocoded.xlsx`.
- **Graph:** You already have `dallas_drive.graphml`. Reuse it — don't re-download.
- **Column name:** The bid price column is `ENTREE_BID_PRICE` (same as X9). ✓


## Section 1 — Starter Scaffold

The cell below gives you a working starting point.
Complete the `# TODO` sections as a group.

You are **not** required to follow this structure — if your group has a better
approach, use it. But make sure your final notebook covers all rubric categories.


In [28]:
import os, time
from pathlib import Path
import pandas as pd
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import AntPath
from geopy.geocoders import Nominatim

# ── Constants ─────────────────────────────────────────────────────────────────
SUPPLY_COST_PER_ENTREE = 52.00   # presentation-day supply cost per entree
DAILY_SUPPLY_CAP       = 427     # presentation-day supply cap
METERS_PER_MILE        = 1609.34

# Data folder auto-detection (works if notebook is launched outside Downloads)
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'restaurants_presentation.xlsx').exists():
    downloads_dir = Path.home() / 'Downloads'
    if (downloads_dir / 'restaurants_presentation.xlsx').exists():
        PROJECT_DIR = downloads_dir

# Company headquarters — all routes start and end here
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'   # Dallas Love Field

print(f'Supply cost per entree: ${SUPPLY_COST_PER_ENTREE:.2f}')
print(f'Daily supply cap:       {DAILY_SUPPLY_CAP} entrees')
print(f'HQ address:             {HQ_ADDRESS}')
print(f'Project directory:      {PROJECT_DIR}')

Supply cost per entree: $52.00
Daily supply cap:       427 entrees
HQ address:             8333 George Coker Cir, Dallas, TX
Project directory:      /Users/macbookair/Desktop/SMU/ITOM 3355/Project One - Group Challenge/Cuy-Delivery


### Geocoding HQ
The cell below geocodes HQ and snaps it to the road network.
Run it once — the result is stored in `HQ_NODE` and reused by `build_route()`.
If you already have the cached graph from X10D1, HQ geocoding takes about 2 seconds.


In [9]:
# ── Geocode HQ (run once; result stored in HQ_NODE after graph loads) ──────────
HQ_GEO_FILE = PROJECT_DIR / 'hq_geocoded.csv'

if HQ_GEO_FILE.exists():
    hq_row  = pd.read_csv(HQ_GEO_FILE).iloc[0]
    HQ_LAT  = float(hq_row['LATITUDE'])
    HQ_LON  = float(hq_row['LONGITUDE'])
    print(f'HQ coords loaded from cache: ({HQ_LAT:.5f}, {HQ_LON:.5f})')
else:
    print('Geocoding HQ address...')
    _geo = Nominatim(user_agent='cuy-delivery-hq')
    time.sleep(2)
    _loc = _geo.geocode(HQ_ADDRESS, timeout=10)
    assert _loc is not None, f'Could not geocode HQ: {HQ_ADDRESS}'
    HQ_LAT, HQ_LON = _loc.latitude, _loc.longitude
    pd.DataFrame([{'ADDRESS': HQ_ADDRESS,
                   'LATITUDE': HQ_LAT,
                   'LONGITUDE': HQ_LON}]).to_csv(HQ_GEO_FILE, index=False)
    print(f'HQ geocoded and cached: ({HQ_LAT:.5f}, {HQ_LON:.5f})')

# HQ_NODE is set after the graph loads (see code_graph cell)

HQ coords loaded from cache: (32.85382, -96.84703)


In [29]:
# ── Load restaurant data ───────────────────────────────────────────────────────
def find_input_file(filename_candidates):
    """Return first matching file path from common folders."""
    if isinstance(filename_candidates, str):
        filename_candidates = [filename_candidates]

    search_roots = [
        PROJECT_DIR,
        Path.cwd(),
        Path.home() / 'Downloads',
        Path.home() / 'Desktop',
        Path.home() / 'Documents',
    ]

    seen = set()
    for root in search_roots:
        root = root.resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)

        for fname in filename_candidates:
            candidate = root / fname
            if candidate.exists():
                return candidate
    return None

# Prefer project files, but gracefully fall back to X9 practice files
restaurants_file = find_input_file(['restaurants_presentation.xlsx', 'restaurants_full.xlsx', 'restaurants_x9.xlsx', 'restaurants_x9 (1).xlsx'])
trucks_file = find_input_file(['trucks_presentation.csv', 'trucks.csv'])

assert restaurants_file is not None, (
    'Could not find restaurant file. Expected restaurants_presentation.xlsx (or a fallback restaurant workbook).'
)

# Use the discovered folder as project directory for caches/graph files
PROJECT_DIR = restaurants_file.parent

print(f'Using data directory: {PROJECT_DIR}')
print(f'Restaurant source:   {restaurants_file.name}')

df_raw = pd.read_excel(restaurants_file)

# Normalize X9 column names to current project names if needed
rename_map = {
    'ENTREE_PRICE_BID': 'ENTREE_BID_PRICE',
}
df_raw = df_raw.rename(columns={k: v for k, v in rename_map.items() if k in df_raw.columns})

required_rest_cols = {'RESTAURANT', 'ADDRESS', 'ENTREES_REQUESTED', 'ENTREE_BID_PRICE'}
missing_rest_cols = required_rest_cols - set(df_raw.columns)
assert not missing_rest_cols, f'Missing restaurant columns: {missing_rest_cols}'

print(f'Restaurants loaded: {len(df_raw)}')
print(f'Total entrees demanded: {df_raw["ENTREES_REQUESTED"].sum()}')
print(f'Bid price range: ${df_raw["ENTREE_BID_PRICE"].min()} – ${df_raw["ENTREE_BID_PRICE"].max()}')
display(df_raw.head())

# ── Load truck configs ─────────────────────────────────────────────────────────
if trucks_file is not None:
    trucks_df = pd.read_csv(trucks_file)
    print(f'\nTruck source:        {trucks_file.name}')
else:
    print('\n⚠️ trucks.csv not found. Using fallback 5-truck template so notebook can run.')
    trucks_df = pd.DataFrame([
        {'truck_name': 'Sparrow', 'fixed_cost': 75.0,  'per_mile_cost': 2.50, 'capacity': 60},
        {'truck_name': 'Osprey',  'fixed_cost': 120.0, 'per_mile_cost': 2.00, 'capacity': 85},
        {'truck_name': 'Falcon',  'fixed_cost': 180.0, 'per_mile_cost': 1.60, 'capacity': 120},
        {'truck_name': 'Hawk',    'fixed_cost': 240.0, 'per_mile_cost': 1.30, 'capacity': 165},
        {'truck_name': 'Condor',  'fixed_cost': 300.0, 'per_mile_cost': 1.05, 'capacity': 220},
    ])

required_truck_cols = {'truck_name', 'fixed_cost', 'per_mile_cost', 'capacity'}
missing_truck_cols = required_truck_cols - set(trucks_df.columns)
assert not missing_truck_cols, f'Missing truck columns: {missing_truck_cols}'

truck_list = trucks_df.to_dict(orient='records')
print(f'Trucks available:    {len(truck_list)}')
display(trucks_df)

Using data directory: /Users/macbookair/Desktop/SMU/ITOM 3355/Project One - Group Challenge/Cuy-Delivery
Restaurant source:   restaurants_presentation.xlsx
Restaurants loaded: 32
Total entrees demanded: 589
Bid price range: $42 – $95


,RESTAURANT,ADDRESS,ENTREES_REQUESTED,ENTREE_BID_PRICE
0,The French Room and Cuy Porch,"1321 Commerce St, Dallas, TX",12,88
1,Pappas Bros. Steak and Cuy House,"10477 Lombardy Ln, Dallas, TX",12,90
2,Truluck's Ocean's Finest Seafood Crab and Cuy,"2401 McKinney Avenue, Dallas, TX",20,78
3,Al Biernat's Fine Cuy,"4217 Oak Lawn Avenue, Dallas, TX",14,95
4,Sweet Georgia Cuy,"2840 E Ledbetter Dr, Dallas, TX",20,48



Truck source:        trucks_presentation.csv
Trucks available:    5


,truck_name,fixed_cost,per_mile_cost,capacity
0,Kite,55,3.50,75
1,Heron,120,2.25,120
2,Pelican,160,1.50,150
3,Albatross,240,1.25,220
4,Condor,185,2.75,999999


In [30]:
# ── Geocode addresses (cached) ─────────────────────────────────────────────────
geo_cache_name = f"{Path(restaurants_file).stem}_geocoded.xlsx"
GEO_FILE = PROJECT_DIR / geo_cache_name

if GEO_FILE.exists():
    geo_df = pd.read_excel(GEO_FILE)
    print(f'Loaded geocoded coords from {GEO_FILE.name}')
else:
    print(f'Geocoding addresses for {restaurants_file.name} — one-time cache build...')
    geolocator = Nominatim(user_agent='cuy-delivery-project1')

    def geocode_address(address):
        try:
            time.sleep(2)
            loc = geolocator.geocode(address, timeout=10)
            return (loc.latitude, loc.longitude) if loc else (None, None)
        except Exception as e:
            print(f'  Error: {address} — {e}')
            return (None, None)

    geo_df = df_raw[['RESTAURANT', 'ADDRESS']].copy()
    geo_df['LATITUDE'], geo_df['LONGITUDE'] = zip(
        *geo_df['ADDRESS'].apply(geocode_address)
    )
    geo_df.to_excel(GEO_FILE, index=False)
    print(f'Geocoding complete — saved to {GEO_FILE.name}')

df_raw = df_raw.merge(geo_df[['RESTAURANT', 'LATITUDE', 'LONGITUDE']],
                      on='RESTAURANT', how='left')

# Sanity check
bad = df_raw[df_raw['LATITUDE'].isna() | df_raw['LONGITUDE'].isna()]
if not bad.empty:
    print(f'⚠️  {len(bad)} restaurants failed geocoding:')
    display(bad[['RESTAURANT', 'ADDRESS']])
else:
    print(f'✅ All {len(df_raw)} restaurants geocoded successfully')

Geocoding addresses for restaurants_presentation.xlsx — one-time cache build...
Geocoding complete — saved to restaurants_presentation_geocoded.xlsx
✅ All 32 restaurants geocoded successfully


In [31]:
# ── Load road network graph (reuse existing file if present) ───────────────────
GRAPH_FILE = PROJECT_DIR / 'dallas_drive.graphml'

if GRAPH_FILE.exists():
    G = ox.load_graphml(GRAPH_FILE)
    print(f'Graph loaded: {len(G.nodes):,} nodes, {len(G.edges):,} edges')
else:
    print('Graph not found — downloading from OpenStreetMap (3–5 min, one time only)...')
    origin = (df_raw['LATITUDE'].mean(), df_raw['LONGITUDE'].mean())
    ox.settings.use_cache = True
    G = ox.graph_from_point(origin, dist=32186, network_type='drive', simplify=True)
    ox.save_graphml(G, filepath=GRAPH_FILE)
    print(f'Graph saved to {GRAPH_FILE}')

# Refresh osmnx.distance so optional deps discovered after package installs.
import importlib
import osmnx.distance as _oxdist
_oxdist = importlib.reload(_oxdist)

# Snap restaurants/HQ to nearest graph nodes.
# Robust fallback: if BallTree is unavailable for unprojected graphs,
# project graph + points and use projected nearest-node search.
try:
    df_raw['NODE_ID'] = _oxdist.nearest_nodes(
        G,
        X=df_raw['LONGITUDE'].to_numpy(),
        Y=df_raw['LATITUDE'].to_numpy()
    )
    HQ_NODE = _oxdist.nearest_nodes(G, HQ_LON, HQ_LAT)
except ImportError:
    import geopandas as gpd

    print('BallTree dependency unavailable; using projected-graph nearest-node fallback...')
    G_proj = ox.project_graph(G)

    rest_pts = gpd.GeoDataFrame(
        df_raw[['LONGITUDE', 'LATITUDE']].copy(),
        geometry=gpd.points_from_xy(df_raw['LONGITUDE'], df_raw['LATITUDE']),
        crs='EPSG:4326'
    ).to_crs(G_proj.graph['crs'])

    hq_pt = gpd.GeoDataFrame(
        [{'LONGITUDE': HQ_LON, 'LATITUDE': HQ_LAT}],
        geometry=gpd.points_from_xy([HQ_LON], [HQ_LAT]),
        crs='EPSG:4326'
    ).to_crs(G_proj.graph['crs'])

    df_raw['NODE_ID'] = _oxdist.nearest_nodes(
        G_proj,
        X=rest_pts.geometry.x.to_numpy(),
        Y=rest_pts.geometry.y.to_numpy()
    )
    HQ_NODE = _oxdist.nearest_nodes(
        G_proj,
        float(hq_pt.geometry.x.iloc[0]),
        float(hq_pt.geometry.y.iloc[0])
    )

print(f'HQ snapped to node: {HQ_NODE}')

# Add margin columns
df_raw['NET_MARGIN']      = df_raw['ENTREE_BID_PRICE'] - SUPPLY_COST_PER_ENTREE
df_raw['GROSS_POTENTIAL'] = df_raw['NET_MARGIN'] * df_raw['ENTREES_REQUESTED']

print('\nRestaurants by net margin (descending):')
display(df_raw[['RESTAURANT', 'ENTREES_REQUESTED', 'ENTREE_BID_PRICE',
                'NET_MARGIN', 'GROSS_POTENTIAL']].sort_values('NET_MARGIN', ascending=False))

Graph not found — downloading from OpenStreetMap (3–5 min, one time only)...
Graph saved to /Users/macbookair/Desktop/SMU/ITOM 3355/Project One - Group Challenge/Cuy-Delivery/dallas_drive.graphml
HQ snapped to node: 4495043101

Restaurants by net margin (descending):


,RESTAURANT,ENTREES_REQUESTED,ENTREE_BID_PRICE,NET_MARGIN,GROSS_POTENTIAL
3,Al Biernat's Fine Cuy,14,95,43.0,602.0
5,The Capital Cuy,15,92,40.0,600.0
1,Pappas Bros. Steak and Cuy House,12,90,38.0,456.0
0,The French Room and Cuy Porch,12,88,36.0,432.0
19,Al Biernat's Fine Cuy North,14,88,36.0,504.0
11,Pyramid Contemporary Cuy,10,85,33.0,330.0
15,La Stella Cuyhouse,18,84,32.0,576.0
6,Café South Pacific Cuy,16,82,30.0,480.0
14,Tei Tei Cuy Robata,15,80,28.0,420.0
10,Gorji Cuy,16,78,26.0,416.0


In [32]:
# ── Core functions from X10D1 — updated for HQ-anchored routes ───────────────
from itertools import combinations

# Distance cache makes repeated scenario testing much faster
DIST_CACHE = {}

def _shortest_path_length_m(G, start_node, end_node):
    """Return shortest-path road distance in meters, cached."""
    key = (int(start_node), int(end_node))
    if key in DIST_CACHE:
        return DIST_CACHE[key]
    try:
        dist_m = nx.shortest_path_length(G, key[0], key[1], weight='length')
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        dist_m = float('inf')
    DIST_CACHE[key] = float(dist_m)
    return DIST_CACHE[key]


def allocate_supply(df, supply_cap, truck_capacity):
    """Allocate entrees by highest margin first.

    Respects both daily supply and truck capacity.
    Partial fills are allowed.
    """
    required_cols = {
        'RESTAURANT', 'ADDRESS', 'ENTREES_REQUESTED', 'ENTREE_BID_PRICE',
        'NET_MARGIN', 'NODE_ID', 'LATITUDE', 'LONGITUDE'
    }
    missing = required_cols - set(df.columns)
    assert not missing, f'Missing columns for allocation: {missing}'

    effective_cap = int(min(supply_cap, truck_capacity))
    remaining = effective_cap

    # Sort by gross profit (margin × quantity) first
    df_ranked = df.copy()
    df_ranked['GROSS_POTENTIAL'] = df_ranked['NET_MARGIN'] * df_ranked['ENTREES_REQUESTED']
    ranked = df_ranked.sort_values(
        ['GROSS_POTENTIAL', 'NET_MARGIN', 'ENTREE_BID_PRICE'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    rows = []
    for _, r in ranked.iterrows():
        if remaining <= 0:
            break

        requested = int(r['ENTREES_REQUESTED'])
        allocated = min(requested, remaining)
        if allocated <= 0:
            continue

        revenue = allocated * float(r['ENTREE_BID_PRICE'])
        supply_cost = allocated * SUPPLY_COST_PER_ENTREE

        rows.append({
            'RESTAURANT': r['RESTAURANT'],
            'ADDRESS': r['ADDRESS'],
            'LATITUDE': float(r['LATITUDE']),
            'LONGITUDE': float(r['LONGITUDE']),
            'NODE_ID': int(r['NODE_ID']),
            'ENTREES_REQUESTED': requested,
            'ENTREE_BID_PRICE': float(r['ENTREE_BID_PRICE']),
            'NET_MARGIN': float(r['NET_MARGIN']),
            'ALLOCATED_ENTREES': int(allocated),
            'REVENUE': float(revenue),
            'SUPPLY_COST': float(supply_cost),
            'GROSS_PROFIT': float(revenue - supply_cost),
        })

        remaining -= allocated

    alloc_df = pd.DataFrame(rows)
    if alloc_df.empty:
        return pd.DataFrame(columns=[
            'RESTAURANT', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'NODE_ID',
            'ENTREES_REQUESTED', 'ENTREE_BID_PRICE', 'NET_MARGIN',
            'ALLOCATED_ENTREES', 'REVENUE', 'SUPPLY_COST', 'GROSS_PROFIT'
        ])

    assert alloc_df['ALLOCATED_ENTREES'].sum() <= effective_cap, 'Over-allocated supply'
    return alloc_df


def build_route(alloc_df, G, hq_node):
    """Build greedy nearest-neighbor route anchored at HQ.

    Includes HQ → first stop and final stop → HQ in total miles.
    """
    route_cols = list(alloc_df.columns) + [
        'STOP_NUM', 'DISTANCE_FROM_PREV_M', 'DISTANCE_FROM_PREV_MI', 'CUMULATIVE_MILES'
    ]

    if alloc_df.empty:
        empty = pd.DataFrame(columns=route_cols)
        empty.attrs['RETURN_TO_HQ_M'] = 0.0
        empty.attrs['TOTAL_METERS'] = 0.0
        empty.attrs['TOTAL_MILES'] = 0.0
        return empty

    pending = alloc_df.copy().reset_index(drop=True)
    current = int(hq_node)
    cumulative_m = 0.0
    stop_num = 1
    route_rows = []

    while len(pending) > 0:
        pending = pending.copy()
        pending['LEG_M'] = pending['NODE_ID'].apply(lambda n: _shortest_path_length_m(G, current, n))
        pending = pending.sort_values(['LEG_M', 'NET_MARGIN', 'ENTREE_BID_PRICE'],
                                      ascending=[True, False, False]).reset_index(drop=True)

        nxt = pending.iloc[0].copy()
        leg_m = float(nxt['LEG_M'])
        assert leg_m < float('inf'), f'No route from node {current} to next stop'

        cumulative_m += leg_m
        nxt['STOP_NUM'] = int(stop_num)
        nxt['DISTANCE_FROM_PREV_M'] = leg_m
        nxt['DISTANCE_FROM_PREV_MI'] = leg_m / METERS_PER_MILE
        nxt['CUMULATIVE_MILES'] = cumulative_m / METERS_PER_MILE
        route_rows.append(nxt.drop(labels=['LEG_M']))

        current = int(nxt['NODE_ID'])
        pending = pending.iloc[1:].reset_index(drop=True)
        stop_num += 1

    route_df = pd.DataFrame(route_rows)

    return_to_hq_m = _shortest_path_length_m(G, current, hq_node)
    assert return_to_hq_m < float('inf'), f'No return route from node {current} to HQ'

    total_m = cumulative_m + return_to_hq_m
    route_df.attrs['RETURN_TO_HQ_M'] = float(return_to_hq_m)
    route_df.attrs['TOTAL_METERS'] = float(total_m)
    route_df.attrs['TOTAL_MILES'] = float(total_m / METERS_PER_MILE)

    return route_df


def calculate_pnl(route_df, truck):
    """Calculate full P&L for one truck + one route."""
    entrees = int(route_df['ALLOCATED_ENTREES'].sum()) if not route_df.empty else 0
    revenue = float(route_df['REVENUE'].sum()) if not route_df.empty else 0.0
    supply_cost = float(route_df['SUPPLY_COST'].sum()) if not route_df.empty else 0.0
    gross_profit = float(revenue - supply_cost)

    if 'TOTAL_MILES' in route_df.attrs:
        miles = float(route_df.attrs['TOTAL_MILES'])
    elif not route_df.empty:
        miles = float(route_df['CUMULATIVE_MILES'].iloc[-1])
    else:
        miles = 0.0

    truck_fixed = float(truck['fixed_cost'])
    truck_variable = float(miles * float(truck['per_mile_cost']))
    net_profit = float(gross_profit - truck_fixed - truck_variable)

    return {
        'truck_name': truck['truck_name'],
        'entrees': entrees,
        'miles': miles,
        'revenue': revenue,
        'supply_cost': supply_cost,
        'gross_profit': gross_profit,
        'truck_fixed_cost': truck_fixed,
        'truck_variable_cost': truck_variable,
        'net_profit': net_profit,
        'return_to_hq_miles': float(route_df.attrs.get('RETURN_TO_HQ_M', 0.0) / METERS_PER_MILE),
    }


def evaluate_single_truck(df_subset, truck, supply_cap, G, hq_node):
    """Convenience wrapper to run allocation → route → P&L."""
    alloc = allocate_supply(df_subset, supply_cap, truck['capacity'])
    route = build_route(alloc, G, hq_node)
    pnl = calculate_pnl(route, truck)
    return {
        'alloc': alloc,
        'route': route,
        'pnl': pnl,
    }


print('Functions defined and ready for scenario optimization.')

Functions defined and ready for scenario optimization.


In [33]:
# ── Kruskal (Minimum Spanning Tree) utilities ─────────────────────────────────
import math


def kruskal_mst(nodes, weighted_edges):
    """Return MST edges + total weight using Kruskal's algorithm.

    Parameters
    ----------
    nodes : list[int]
        Node IDs included in the graph.
    weighted_edges : list[tuple[int, int, float]]
        Candidate undirected edges in the form (u, v, weight).
    """
    parent = {n: n for n in nodes}
    rank = {n: 0 for n in nodes}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]  # path compression
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1
        return True

    mst_edges = []
    total_weight = 0.0

    for u, v, w in sorted(weighted_edges, key=lambda e: e[2]):
        if union(u, v):
            mst_edges.append((u, v, float(w)))
            total_weight += float(w)
            if len(mst_edges) == max(0, len(nodes) - 1):
                break

    # If graph is disconnected, MST cannot span all nodes.
    if len(nodes) > 0 and len(mst_edges) != len(nodes) - 1:
        return [], float('inf')

    return mst_edges, float(total_weight)


def build_complete_weighted_edges(node_ids, G):
    """Create pairwise road-distance edges for Kruskal."""
    edges = []
    unique_nodes = list(dict.fromkeys(int(n) for n in node_ids))

    for i, u in enumerate(unique_nodes):
        for v in unique_nodes[i + 1:]:
            d = _shortest_path_length_m(G, u, v)
            if math.isfinite(d):
                edges.append((u, v, float(d)))

    return unique_nodes, edges


def summarize_kruskal_for_alloc(alloc_df, G, hq_node):
    """Run Kruskal on {HQ + allocated stops} and summarize MST length."""
    if alloc_df.empty:
        return {
            'mst_edges': pd.DataFrame(columns=['u', 'v', 'distance_m', 'distance_mi']),
            'mst_total_m': 0.0,
            'mst_total_mi': 0.0,
        }

    node_ids = [int(hq_node)] + alloc_df['NODE_ID'].astype(int).tolist()
    nodes, weighted_edges = build_complete_weighted_edges(node_ids, G)
    mst_edges, total_m = kruskal_mst(nodes, weighted_edges)

    mst_df = pd.DataFrame(mst_edges, columns=['u', 'v', 'distance_m'])
    if not mst_df.empty:
        mst_df['distance_mi'] = mst_df['distance_m'] / METERS_PER_MILE
    else:
        mst_df = pd.DataFrame(columns=['u', 'v', 'distance_m', 'distance_mi'])

    return {
        'mst_edges': mst_df,
        'mst_total_m': float(total_m),
        'mst_total_mi': float(total_m / METERS_PER_MILE) if math.isfinite(total_m) else float('inf'),
    }


print("Kruskal utilities added: kruskal_mst(), build_complete_weighted_edges(), summarize_kruskal_for_alloc().")

Kruskal utilities added: kruskal_mst(), build_complete_weighted_edges(), summarize_kruskal_for_alloc().


## Section 2 — Single-Truck Scenarios

Run each truck as a single-truck deployment. Build a comparison table.
Which truck produces the highest net profit when operating alone?


In [34]:
# ── Run all 5 single-truck scenarios ──────────────────────────────────────────
single_records = []
single_details = {}

for truck in truck_list:
    result = evaluate_single_truck(df_raw, truck, DAILY_SUPPLY_CAP, G, HQ_NODE)
    single_details[truck['truck_name']] = result
    pnl = result['pnl']

    single_records.append({
        'scenario': f"single::{truck['truck_name']}",
        'truck': truck['truck_name'],
        'capacity': int(truck['capacity']),
        'fixed_cost': float(truck['fixed_cost']),
        'per_mile': float(truck['per_mile_cost']),
        'stops': int(len(result['route'])),
        'entrees': int(pnl['entrees']),
        'miles': round(float(pnl['miles']), 2),
        'revenue': float(pnl['revenue']),
        'supply_cost': float(pnl['supply_cost']),
        'truck_cost': float(pnl['truck_fixed_cost'] + pnl['truck_variable_cost']),
        'net_profit': float(pnl['net_profit']),
    })

single_df = pd.DataFrame(single_records).sort_values('net_profit', ascending=False).reset_index(drop=True)

print('=== Single-Truck Comparison @ current DAILY_SUPPLY_CAP ===')
display(single_df)

best_single_name = single_df.iloc[0]['truck']
best_single_profit = single_df.iloc[0]['net_profit']
print(f"\nBest single-truck option: {best_single_name} (${best_single_profit:,.2f})")

# quick look at the most important served stops for the top single-truck winner
best_single_route = single_details[best_single_name]['route'].copy()
if not best_single_route.empty:
    preview_cols = ['STOP_NUM', 'RESTAURANT', 'ALLOCATED_ENTREES', 'NET_MARGIN', 'DISTANCE_FROM_PREV_MI']
    print('\nTop single-truck route preview:')
    display(best_single_route[preview_cols].head(10))

=== Single-Truck Comparison @ current DAILY_SUPPLY_CAP ===


,scenario,truck,capacity,fixed_cost,per_mile,stops,entrees,miles,revenue,supply_cost,truck_cost,net_profit
0,single::Condor,Condor,999999,185.0,2.75,25,427,77.30,31460.0,22204.0,397.573463,8858.426537
1,single::Albatross,Albatross,220,240.0,1.25,13,220,58.45,17737.0,11440.0,313.068168,5983.931832
2,single::Pelican,Pelican,150,160.0,1.50,9,150,42.09,12454.0,7800.0,223.138480,4430.861520
3,single::Heron,Heron,120,120.0,2.25,7,120,34.05,9862.0,6240.0,196.612860,3425.387140
4,single::Kite,Kite,75,55.0,3.50,4,75,33.12,6154.0,3900.0,170.935413,2083.064587



Best single-truck option: Condor ($8,858.43)

Top single-truck route preview:


,STOP_NUM,RESTAURANT,ALLOCATED_ENTREES,NET_MARGIN,DISTANCE_FROM_PREV_MI
0,1,The Original Pancuy House,15,14.0,1.363295
0,2,True Cuy Kitchen,20,18.0,2.146215
0,3,Café South Pacific Cuy,16,30.0,2.132278
0,4,Al Biernat's Fine Cuy,14,43.0,1.246669
0,5,Cuy de Brazil,25,6.0,1.364926
0,6,Fogo de Cuy,22,16.0,0.447607
0,7,Truluck's Ocean's Finest Seafood Crab and Cuy,20,26.0,0.155695
0,8,Hotel Crescent Cuy,18,22.0,0.358271
0,9,The Capital Cuy,15,40.0,0.028708
0,10,Pyramid Contemporary Cuy,10,33.0,0.747567


In [19]:
# ── Kruskal metrics for single-truck table ─────────────────────────────────────
single_kruskal_rows = []
single_kruskal_details = {}

for truck_name, detail in single_details.items():
    k = summarize_kruskal_for_alloc(detail['alloc'], G, HQ_NODE)
    single_kruskal_details[truck_name] = k

    route_miles = float(detail['pnl']['miles'])
    mst_miles = float(k['mst_total_mi'])
    single_kruskal_rows.append({
        'truck': truck_name,
        'mst_miles': mst_miles,
        'route_minus_mst_miles': route_miles - mst_miles,
        'route_to_mst_ratio': (route_miles / mst_miles) if mst_miles > 0 else float('nan')
    })

single_kruskal_df = pd.DataFrame(single_kruskal_rows)

single_df = (
    single_df.drop(columns=['mst_miles', 'route_minus_mst_miles', 'route_to_mst_ratio'], errors='ignore')
             .merge(single_kruskal_df, on='truck', how='left')
             .sort_values('net_profit', ascending=False)
             .reset_index(drop=True)
)

print('=== Single-Truck Comparison (with Kruskal MST metrics) ===')
display(single_df)


=== Single-Truck Comparison (with Kruskal MST metrics) ===


,scenario,truck,capacity,fixed_cost,per_mile,stops,entrees,miles,revenue,supply_cost,truck_cost,net_profit,mst_miles,route_minus_mst_miles,route_to_mst_ratio
0,single::Hawk,Hawk,165,240.0,1.30,10,165,43.72,20210.0,4620.0,296.834529,15293.165471,27.228169,16.490699,1.605648
1,single::Condor,Condor,220,300.0,1.05,10,165,43.72,20210.0,4620.0,345.904812,15244.095188,27.228169,16.490699,1.605648
2,single::Falcon,Falcon,120,180.0,1.60,8,120,41.57,16140.0,3360.0,246.507424,12533.492576,26.881270,14.685869,1.546323
3,single::Osprey,Osprey,85,120.0,2.00,6,85,30.23,12400.0,2380.0,180.453942,9839.546058,18.096431,12.130540,1.670328
4,single::Sparrow,Sparrow,60,75.0,2.50,5,60,28.35,9310.0,1680.0,145.874744,7484.125256,16.733135,11.616762,1.694237


## Section 3 — Two-Truck Scenarios (Optional Extension)

If you choose to explore a two-truck deployment, design your restaurant split
strategy here. Each group will approach this differently — document your thinking.

**Key question:** Does splitting the 29 restaurants across two smaller trucks
beat running one large truck?

Think about:
- Which restaurants cluster geographically? (Check the map from X10D1)
- Which truck combination minimizes total fixed cost while preserving capacity?
- Does the supply cap (427 total entrees) change which restaurants you prioritize?


In [20]:
# ── Two-truck split strategies (unique approach) ───────────────────────────────
# We evaluate multiple split philosophies, then let profit decide.

df_split_base = df_raw.copy()
df_split_base['DIST_HQ_M'] = df_split_base['NODE_ID'].apply(lambda n: _shortest_path_length_m(G, HQ_NODE, n))
df_split_base['DIST_HQ_MI'] = df_split_base['DIST_HQ_M'] / METERS_PER_MILE
df_split_base['MARGIN_DENSITY'] = df_split_base['NET_MARGIN'] / (df_split_base['DIST_HQ_MI'] + 0.75)


def _sanitize_split(left_df, right_df, min_stops=3):
    left_df = left_df.drop_duplicates(subset=['RESTAURANT']).copy()
    right_df = right_df.drop_duplicates(subset=['RESTAURANT']).copy()

    overlap = set(left_df['RESTAURANT']) & set(right_df['RESTAURANT'])
    assert len(overlap) == 0, f'Split overlap detected: {overlap}'

    if len(left_df) < min_stops or len(right_df) < min_stops:
        return None
    return left_df, right_df


def split_latitude(df):
    med = df['LATITUDE'].median()
    return _sanitize_split(df[df['LATITUDE'] >= med], df[df['LATITUDE'] < med])


def split_longitude(df):
    med = df['LONGITUDE'].median()
    return _sanitize_split(df[df['LONGITUDE'] >= med], df[df['LONGITUDE'] < med])


def split_margin_tier(df):
    ranked = df.sort_values(['NET_MARGIN', 'ENTREE_BID_PRICE'], ascending=[False, False]).reset_index(drop=True)
    left = ranked.iloc[::2].copy()
    right = ranked.iloc[1::2].copy()
    return _sanitize_split(left, right)


def split_profit_density_snake(df):
    ranked = df.sort_values(['MARGIN_DENSITY', 'NET_MARGIN'], ascending=[False, False]).reset_index(drop=True)
    left_rows, right_rows = [], []
    left_load, right_load = 0, 0

    for _, row in ranked.iterrows():
        demand = int(row['ENTREES_REQUESTED'])
        if left_load <= right_load:
            left_rows.append(row)
            left_load += demand
        else:
            right_rows.append(row)
            right_load += demand

    left = pd.DataFrame(left_rows)
    right = pd.DataFrame(right_rows)
    return _sanitize_split(left, right)


strategy_builders = {
    'geo_latitude_split': split_latitude,
    'geo_longitude_split': split_longitude,
    'margin_tier_alternating': split_margin_tier,
    'profit_density_snake': split_profit_density_snake,
}

candidate_splits = {}
for strategy_name, fn in strategy_builders.items():
    split_result = fn(df_split_base)
    if split_result is not None:
        candidate_splits[strategy_name] = split_result

print('Candidate split strategies ready:')
for k, (d1, d2) in candidate_splits.items():
    print(f"- {k}: {len(d1)} restaurants vs {len(d2)} restaurants")

Candidate split strategies ready:
- geo_latitude_split: 5 restaurants vs 5 restaurants
- geo_longitude_split: 5 restaurants vs 5 restaurants
- margin_tier_alternating: 5 restaurants vs 5 restaurants
- profit_density_snake: 5 restaurants vs 5 restaurants


In [35]:
# ── Two-truck optimizer + 1-vs-2 auto-selection ───────────────────────────────
from IPython.display import Markdown


def optimize_two_truck_for_supply(supply_cap, candidate_splits):
    """Evaluate all truck pairs, split strategies, and feasible cap splits."""
    rows = []

    for truck1, truck2 in combinations(truck_list, 2):
        c1_max = int(truck1['capacity'])
        c2_max = int(truck2['capacity'])

        # feasible cap1 range given cap2 and total supply
        min_cap1 = max(0, int(supply_cap - c2_max))
        max_cap1 = min(int(supply_cap), c1_max)

        # evaluate every 5 entrees + boundaries (faster but still precise enough)
        cap1_candidates = set([min_cap1, max_cap1])
        cap1_candidates.update(range(min_cap1, max_cap1 + 1, 5))

        for strategy_name, (df1_raw, df2_raw) in candidate_splits.items():
            for cap1 in sorted(cap1_candidates):
                cap2 = int(supply_cap - cap1)
                if cap2 < 0 or cap2 > c2_max:
                    continue

                r1 = evaluate_single_truck(df1_raw, truck1, cap1, G, HQ_NODE)
                r2 = evaluate_single_truck(df2_raw, truck2, cap2, G, HQ_NODE)

                # two-truck rule: each truck serves at least 3 restaurants
                if len(r1['route']) < 3 or len(r2['route']) < 3:
                    continue

                entrees_total = int(r1['pnl']['entrees'] + r2['pnl']['entrees'])
                if entrees_total > supply_cap:
                    continue

                net_total = float(r1['pnl']['net_profit'] + r2['pnl']['net_profit'])
                miles_total = float(r1['pnl']['miles'] + r2['pnl']['miles'])

                rows.append({
                    'config_type': 'two_truck',
                    'supply_cap': int(supply_cap),
                    'strategy': strategy_name,
                    'truck_1': truck1['truck_name'],
                    'truck_2': truck2['truck_name'],
                    'cap_1': int(cap1),
                    'cap_2': int(cap2),
                    'stops_1': int(len(r1['route'])),
                    'stops_2': int(len(r2['route'])),
                    'entrees_1': int(r1['pnl']['entrees']),
                    'entrees_2': int(r2['pnl']['entrees']),
                    'entrees_total': entrees_total,
                    'miles_total': miles_total,
                    'net_profit': net_total,
                    'detail_1': r1,
                    'detail_2': r2,
                })

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values('net_profit', ascending=False).reset_index(drop=True)


def best_single_for_supply(supply_cap):
    rows = []
    details = {}
    for truck in truck_list:
        result = evaluate_single_truck(df_raw, truck, supply_cap, G, HQ_NODE)
        details[truck['truck_name']] = result
        rows.append({
            'config_type': 'single_truck',
            'supply_cap': int(supply_cap),
            'truck': truck['truck_name'],
            'entrees_total': int(result['pnl']['entrees']),
            'miles_total': float(result['pnl']['miles']),
            'net_profit': float(result['pnl']['net_profit']),
            'detail': result,
        })
    single_table = pd.DataFrame(rows).sort_values('net_profit', ascending=False).reset_index(drop=True)
    return single_table.iloc[0].to_dict(), single_table


supply_test_values = sorted(set([100, DAILY_SUPPLY_CAP]))
all_rows = []
best_single_by_supply = {}
best_two_by_supply = {}
final_choice_by_supply = {}

for s in supply_test_values:
    best_single_row, single_table_s = best_single_for_supply(s)
    best_single_by_supply[s] = best_single_row

    best_two_table_s = optimize_two_truck_for_supply(s, candidate_splits)
    if not best_two_table_s.empty:
        best_two_row = best_two_table_s.iloc[0].to_dict()
        best_two_by_supply[s] = best_two_row
    else:
        best_two_row = None

    # row for comparison table
    all_rows.append({
        'supply_cap': int(s),
        'scenario_label': f"single::{best_single_row['truck']}",
        'config_type': 'single_truck',
        'net_profit': float(best_single_row['net_profit']),
        'miles_total': float(best_single_row['miles_total']),
        'entrees_total': int(best_single_row['entrees_total']),
        'notes': 'best single-truck option'
    })

    if best_two_row is not None:
        all_rows.append({
            'supply_cap': int(s),
            'scenario_label': f"two::{best_two_row['truck_1']}+{best_two_row['truck_2']}::{best_two_row['strategy']}",
            'config_type': 'two_truck',
            'net_profit': float(best_two_row['net_profit']),
            'miles_total': float(best_two_row['miles_total']),
            'entrees_total': int(best_two_row['entrees_total']),
            'notes': f"caps {best_two_row['cap_1']}/{best_two_row['cap_2']}"
        })

    if best_two_row is None or best_single_row['net_profit'] >= best_two_row['net_profit']:
        final_choice_by_supply[s] = {
            'winner_type': 'single_truck',
            'winner_name': best_single_row['truck'],
            'net_profit': float(best_single_row['net_profit']),
            'miles_total': float(best_single_row['miles_total']),
            'entrees_total': int(best_single_row['entrees_total']),
            'detail': best_single_row['detail'],
            'supply_cap': int(s),
        }
    else:
        final_choice_by_supply[s] = {
            'winner_type': 'two_truck',
            'winner_name': f"{best_two_row['truck_1']} + {best_two_row['truck_2']}",
            'net_profit': float(best_two_row['net_profit']),
            'miles_total': float(best_two_row['miles_total']),
            'entrees_total': int(best_two_row['entrees_total']),
            'strategy': best_two_row['strategy'],
            'cap_1': int(best_two_row['cap_1']),
            'cap_2': int(best_two_row['cap_2']),
            'detail_1': best_two_row['detail_1'],
            'detail_2': best_two_row['detail_2'],
            'truck_1': best_two_row['truck_1'],
            'truck_2': best_two_row['truck_2'],
            'supply_cap': int(s),
        }

scenario_comparison_df = pd.DataFrame(all_rows).sort_values(
    ['supply_cap', 'net_profit'], ascending=[True, False]
).reset_index(drop=True)

print('=== Scenario Comparison (at least 3 scenarios, 2 supply caps) ===')
display(scenario_comparison_df)

winner_summary_df = pd.DataFrame([
    {
        'supply_cap': s,
        'winner_type': final_choice_by_supply[s]['winner_type'],
        'winner_name': final_choice_by_supply[s]['winner_name'],
        'net_profit': final_choice_by_supply[s]['net_profit'],
        'miles_total': final_choice_by_supply[s]['miles_total'],
        'entrees_total': final_choice_by_supply[s]['entrees_total'],
    }
    for s in sorted(final_choice_by_supply.keys())
])

print('\n=== Auto-Selected Winner by Supply Level ===')
display(winner_summary_df)

# current-day recommendation (for the current DAILY_SUPPLY_CAP)
current_choice = final_choice_by_supply[DAILY_SUPPLY_CAP]

if current_choice['winner_type'] == 'single_truck':
    route = current_choice['detail']['route']
    served_list = ', '.join(route['RESTAURANT'].head(6).tolist())
    recommendation_md = f"""
## Our Recommendation

1. **Recommended configuration:** single truck **{current_choice['winner_name']}**.
2. **Net profit:** **${current_choice['net_profit']:,.2f}** at supply cap **{current_choice['supply_cap']}**.
3. **Miles + entrees:** {current_choice['miles_total']:.2f} miles, {current_choice['entrees_total']} entrees.
4. **Restaurants prioritized:** {served_list} ... (highest-margin-first allocation).
5. **Trade-off:** this option avoids paying a second fixed truck fee while still capturing high-margin demand.
6. **Sensitivity result:** see winner table above; recommendation auto-updates when `DAILY_SUPPLY_CAP` changes.
"""
else:
    route1 = current_choice['detail_1']['route']
    route2 = current_choice['detail_2']['route']
    served1 = ', '.join(route1['RESTAURANT'].head(4).tolist())
    served2 = ', '.join(route2['RESTAURANT'].head(4).tolist())
    recommendation_md = f"""
## Our Recommendation

1. **Recommended configuration:** two trucks **{current_choice['winner_name']}**.
2. **Net profit:** **${current_choice['net_profit']:,.2f}** at supply cap **{current_choice['supply_cap']}**.
3. **Strategy used:** `{current_choice['strategy']}` with supply split {current_choice['cap_1']} / {current_choice['cap_2']}.
4. **Miles + entrees:** {current_choice['miles_total']:.2f} miles total, {current_choice['entrees_total']} entrees served.
5. **Key restaurants:**
   - {current_choice['truck_1']}: {served1} ...
   - {current_choice['truck_2']}: {served2} ...
6. **Sensitivity result:** see winner table above; this framework re-optimizes instantly for new supply caps.
"""

display(Markdown(recommendation_md))
print('Recommendation markdown generated from live results.')

=== Scenario Comparison (at least 3 scenarios, 2 supply caps) ===


,supply_cap,scenario_label,config_type,net_profit,miles_total,entrees_total,notes
0,100,two::Kite+Heron::geo_longitude_split,two_truck,8522.225230,48.113779,100,caps 55/45
1,100,single::Heron,single_truck,2750.592592,33.514404,100,best single-truck option
2,427,two::Heron+Pelican::geo_latitude_split,two_truck,11252.625063,49.482067,165,caps 277/150
3,427,single::Condor,single_truck,8858.426537,77.299441,427,best single-truck option



=== Auto-Selected Winner by Supply Level ===


,supply_cap,winner_type,winner_name,net_profit,miles_total,entrees_total
0,100,two_truck,Kite + Heron,8522.225230,48.113779,100
1,427,two_truck,Heron + Pelican,11252.625063,49.482067,165



## Our Recommendation

1. **Recommended configuration:** two trucks **Heron + Pelican**.
2. **Net profit:** **$11,252.63** at supply cap **427**.
3. **Strategy used:** `geo_latitude_split` with supply split 277 / 150.
4. **Miles + entrees:** 49.48 miles total, 165 entrees served.
5. **Key restaurants:**
   - Heron: The Original Pancuy House, Café South Pacific Cuy, Tei Tei Cuy Robata, Cindi's NY Cuycatesson ...
   - Pelican: Cuy de Brazil, Fogo de Cuy, Truluck's Ocean's Finest Seafood Crab and Cuy, The French Room and Cuy Porch ...
6. **Sensitivity result:** see winner table above; this framework re-optimizes instantly for new supply caps.


Recommendation markdown generated from live results.


In [22]:
# ── Kruskal metrics for best 1-vs-2 scenarios by supply cap ───────────────────
def _mst_miles_from_alloc(alloc_df):
    k = summarize_kruskal_for_alloc(alloc_df, G, HQ_NODE)
    return float(k['mst_total_mi'])

scenario_kruskal_rows = []
for s in sorted(supply_test_values):
    # Best single at this supply
    bs = best_single_by_supply[s]
    bs_mst = _mst_miles_from_alloc(bs['detail']['alloc'])
    scenario_kruskal_rows.append({
        'supply_cap': int(s),
        'scenario_label': f"single::{bs['truck']}",
        'config_type': 'single_truck',
        'net_profit': float(bs['net_profit']),
        'miles_total': float(bs['miles_total']),
        'mst_miles_total': bs_mst,
        'route_minus_mst_miles': float(bs['miles_total']) - bs_mst,
        'entrees_total': int(bs['entrees_total'])
    })

    # Best two-truck at this supply (if feasible)
    if s in best_two_by_supply:
        bt = best_two_by_supply[s]
        bt_mst_1 = _mst_miles_from_alloc(bt['detail_1']['alloc'])
        bt_mst_2 = _mst_miles_from_alloc(bt['detail_2']['alloc'])
        bt_mst_total = bt_mst_1 + bt_mst_2

        scenario_kruskal_rows.append({
            'supply_cap': int(s),
            'scenario_label': f"two::{bt['truck_1']}+{bt['truck_2']}::{bt['strategy']}",
            'config_type': 'two_truck',
            'net_profit': float(bt['net_profit']),
            'miles_total': float(bt['miles_total']),
            'mst_miles_total': bt_mst_total,
            'route_minus_mst_miles': float(bt['miles_total']) - bt_mst_total,
            'entrees_total': int(bt['entrees_total'])
        })

scenario_comparison_with_kruskal_df = (
    pd.DataFrame(scenario_kruskal_rows)
      .sort_values(['supply_cap', 'net_profit'], ascending=[True, False])
      .reset_index(drop=True)
)

print('=== Scenario Comparison (with Kruskal MST metrics) ===')
display(scenario_comparison_with_kruskal_df)

winner_kruskal_rows = []
for s in sorted(final_choice_by_supply.keys()):
    w = final_choice_by_supply[s]
    if w['winner_type'] == 'single_truck':
        mst = _mst_miles_from_alloc(w['detail']['alloc'])
    else:
        mst = _mst_miles_from_alloc(w['detail_1']['alloc']) + _mst_miles_from_alloc(w['detail_2']['alloc'])

    winner_kruskal_rows.append({
        'supply_cap': int(s),
        'winner_type': w['winner_type'],
        'winner_name': w['winner_name'],
        'net_profit': float(w['net_profit']),
        'miles_total': float(w['miles_total']),
        'mst_miles_total': float(mst),
        'route_minus_mst_miles': float(w['miles_total'] - mst),
        'entrees_total': int(w['entrees_total'])
    })

winner_summary_with_kruskal_df = pd.DataFrame(winner_kruskal_rows)
print('\n=== Auto-Selected Winner by Supply Level (with Kruskal MST metrics) ===')
display(winner_summary_with_kruskal_df)


=== Scenario Comparison (with Kruskal MST metrics) ===


,supply_cap,scenario_label,config_type,net_profit,miles_total,mst_miles_total,route_minus_mst_miles,entrees_total
0,100,single::Falcon,single_truck,11052.914163,35.678648,23.883890,11.794758,100
1,100,two::Sparrow+Osprey::geo_latitude_split,two_truck,10991.889910,43.837766,26.931270,16.906496,100
2,165,single::Hawk,single_truck,15293.165471,43.718868,27.228169,16.490699,165
3,165,two::Osprey+Falcon::geo_latitude_split,two_truck,15198.481151,49.482002,29.882275,19.599727,165



=== Auto-Selected Winner by Supply Level (with Kruskal MST metrics) ===


,supply_cap,winner_type,winner_name,net_profit,miles_total,mst_miles_total,route_minus_mst_miles,entrees_total
0,100,single_truck,Falcon,11052.914163,35.678648,23.883890,11.794758,100
1,165,single_truck,Hawk,15293.165471,43.718868,27.228169,16.490699,165


## Section 4 — Validation

Add at least **5 assert statements** validating key constraints and math.
Your notebook should be able to run top-to-bottom with all assertions passing.


In [38]:
# ── Validation assertions ──────────────────────────────────────────────────────
# 1) Single-truck capacity and supply cap checks
for row in single_df.itertuples(index=False):
    truck_obj = next(t for t in truck_list if t['truck_name'] == row.truck)
    assert row.entrees <= int(truck_obj['capacity']), (
        f"{row.truck} over truck capacity: {row.entrees} > {truck_obj['capacity']}"
    )
    assert row.entrees <= DAILY_SUPPLY_CAP, (
        f"{row.truck} exceeded daily cap: {row.entrees} > {DAILY_SUPPLY_CAP}"
    )
print('✅ Single-truck capacity and supply assertions pass')

# 2) P&L identity for best single truck in there
best_single_name = single_df.iloc[0]['truck']
best_single_detail = single_details[best_single_name]
best_single_pnl = best_single_detail['pnl']
assert abs(best_single_pnl['gross_profit'] - (best_single_pnl['revenue'] - best_single_pnl['supply_cost'])) < 1e-6
assert abs(
    best_single_pnl['net_profit'] - (
        best_single_pnl['gross_profit'] - best_single_pnl['truck_fixed_cost'] - best_single_pnl['truck_variable_cost']
    )
) < 1e-6
print('✅ Best single-truck P&L math checks out')

# 3) Route distance includes return-to-HQ leg
if not best_single_detail['route'].empty:
    outbound_only_miles = float(best_single_detail['route']['CUMULATIVE_MILES'].iloc[-1])
    full_miles = float(best_single_pnl['miles'])
    assert full_miles >= outbound_only_miles, 'Return-to-HQ leg missing from miles'
print('✅ HQ return leg is included in route miles')

# 4) If two-truck winner exists at current supply, validate constraints
if DAILY_SUPPLY_CAP in final_choice_by_supply and final_choice_by_supply[DAILY_SUPPLY_CAP]['winner_type'] == 'two_truck':
    c = final_choice_by_supply[DAILY_SUPPLY_CAP]
    r1 = c['detail_1']['route']
    r2 = c['detail_2']['route']
    p1 = c['detail_1']['pnl']
    p2 = c['detail_2']['pnl']

    # no overlap between trucks
    overlap = set(r1['RESTAURANT']) & set(r2['RESTAURANT'])
    assert len(overlap) == 0, f'Restaurants served by both trucks: {overlap}'

    # each truck serves at least 3 restaurants
    assert len(r1) >= 3 and len(r2) >= 3, 'Each truck must serve at least 3 restaurants'

    # combined allocations respect daily cap
    assert int(p1['entrees'] + p2['entrees']) <= DAILY_SUPPLY_CAP, 'Two-truck combined entrees exceed daily cap'

    # combined net equals sum of individual nets
    assert abs(c['net_profit'] - (p1['net_profit'] + p2['net_profit'])) < 1e-6, 'Two-truck net mismatch'

    print('✅ Two-truck constraint assertions pass')
else:
    print('ℹ️ Current winner is single-truck; two-truck assertions skipped for current cap')

# 5) Allocation never exceeds restaurant request in top single scenario
best_alloc = best_single_detail['alloc']
if not best_alloc.empty:
    assert (best_alloc['ALLOCATED_ENTREES'] <= best_alloc['ENTREES_REQUESTED']).all(), (
        'Allocated more entrees than requested at one or more restaurants'
    )
print('✅ Per-restaurant allocation bounds are valid')

print('\nAll required validation checks completed.')

✅ Single-truck capacity and supply assertions pass
✅ Best single-truck P&L math checks out
✅ HQ return leg is included in route miles
✅ Two-truck constraint assertions pass
✅ Per-restaurant allocation bounds are valid

All required validation checks completed.


## ⭐ Final Recommendation (Auto-Generated Above)

The recommendation displayed immediately above this section is **dynamically generated** from the current scenario analysis. 

**It automatically:**
- Uses your current `DAILY_SUPPLY_CAP` setting
- Compares best single-truck vs. best two-truck options
- Reports actual net profit, miles, and allocated restaurants
- Re-calculates instantly if you change supply cap and re-run

**On delivery day (X11D2):** Update `DAILY_SUPPLY_CAP` and re-run the notebook to get your day's recommendation in seconds.


In [37]:
from IPython.display import display, FileLink, HTML
from branca.element import Element
import re


map_export_dir = PROJECT_DIR / 'route_maps'
map_export_dir.mkdir(exist_ok=True)


def slugify(value):
    value = re.sub(r'[^A-Za-z0-9]+', '_', str(value).strip())
    return value.strip('_').lower()


def _route_point_list(route_df):
    if route_df.empty:
        return []
    return route_df[['LATITUDE', 'LONGITUDE']].astype(float).values.tolist()


def build_polished_route_map(route_specs, title, output_name):
    if not route_specs:
        raise ValueError('No routes provided for map export.')

    all_points = [[HQ_LAT, HQ_LON]]
    for spec in route_specs:
        all_points.extend(_route_point_list(spec['route']))
        all_points.append([HQ_LAT, HQ_LON])

    center_lat = sum(pt[0] for pt in all_points) / len(all_points)
    center_lon = sum(pt[1] for pt in all_points) / len(all_points)

    route_map = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=11,
        tiles='CartoDB positron',
        control_scale=True,
    )

    legend_rows = []

    folium.Marker(
        [HQ_LAT, HQ_LON],
        popup=folium.Popup('<b>HQ</b><br>8333 George Coker Cir, Dallas, TX', max_width=250),
        tooltip='HQ',
        icon=folium.Icon(color='red', icon='home', prefix='fa'),
    ).add_to(route_map)

    for spec in route_specs:
        truck_name = spec['truck_name']
        route_df = spec['route'].copy()
        color = spec['color']
        icon_color = spec.get('icon_color', color)
        display_name = spec.get('display_name', truck_name)

        if route_df.empty:
            continue

        path_points = [[HQ_LAT, HQ_LON]] + _route_point_list(route_df) + [[HQ_LAT, HQ_LON]]

        folium.PolyLine(
            path_points,
            color=color,
            weight=6,
            opacity=0.9,
            dash_array='8, 6',
        ).add_to(route_map)

        AntPath(
            path_points,
            color=color,
            pulse_color=icon_color,
            weight=4,
            delay=900,
            dash_array=[12, 24],
        ).add_to(route_map)

        legend_rows.append(
            f"<div style='margin-bottom:4px;'><span style='display:inline-block;width:12px;height:12px;border-radius:50%;background:{color};margin-right:8px;'></span>{display_name}</div>"
        )

        for stop_idx, (_, row) in enumerate(route_df.iterrows(), start=1):
            stop_color = icon_color
            popup_html = f"""
                <div style='font-family:Arial; font-size:13px; min-width:240px;'>
                    <div style='font-size:14px; font-weight:700; margin-bottom:4px;'>Stop {stop_idx}: {row['RESTAURANT']}</div>
                    <div><b>Allocated entrees:</b> {int(row['ALLOCATED_ENTREES'])}</div>
                    <div><b>Net margin:</b> ${float(row['NET_MARGIN']):.2f}</div>
                    <div><b>Gross profit:</b> ${float(row['GROSS_PROFIT']):.2f}</div>
                    <div><b>Cumulative miles:</b> {float(row['CUMULATIVE_MILES']):.2f}</div>
                </div>
            """
            folium.Marker(
                [float(row['LATITUDE']), float(row['LONGITUDE'])],
                tooltip=f"{stop_idx}. {row['RESTAURANT']}",
                popup=folium.Popup(popup_html, max_width=320),
                icon=folium.features.DivIcon(
                    html=f"""
                        <div style='
                            width: 28px; height: 28px; line-height: 26px;
                            border: 2px solid white; border-radius: 50%;
                            background: {stop_color}; color: white;
                            font-size: 12px; font-weight: 700; text-align: center;
                            box-shadow: 0 0 8px rgba(0,0,0,.28);
                        '>{stop_idx}</div>
                    """
                ),
            ).add_to(route_map)

    bounds = [[min(pt[0] for pt in all_points), min(pt[1] for pt in all_points)],
              [max(pt[0] for pt in all_points), max(pt[1] for pt in all_points)]]
    route_map.fit_bounds(bounds)

    legend_html = f"""
    <div style='position: fixed; bottom: 28px; left: 28px; z-index:9999;
                background: rgba(255,255,255,0.95); padding: 12px 14px;
                border: 2px solid #2c3e50; border-radius: 12px;
                box-shadow: 0 2px 12px rgba(0,0,0,.18); font-size: 13px; min-width: 180px;'>
        <div style='font-size:14px; font-weight:700; margin-bottom:8px;'>{title}</div>
        <div style='margin-bottom:4px;'><span style='display:inline-block;width:12px;height:12px;border-radius:50%;background:#e74c3c;margin-right:8px;'></span>HQ</div>
        {''.join(legend_rows)}
        <div style='margin-top:8px; font-size:11px; color:#555;'>Numbered stops show delivery order.</div>
    </div>
    """
    route_map.get_root().html.add_child(Element(legend_html))

    output_path = map_export_dir / output_name
    route_map.save(str(output_path))
    return route_map, output_path


best_single_route_map, best_single_map_path = build_polished_route_map(
    [
        {
            'truck_name': best_single_name,
            'display_name': f'Best single truck: {best_single_name}',
            'route': best_single_route,
            'color': '#1f77b4',
            'icon_color': '#1f77b4',
        }
    ],
    title=f'Best Single-Truck Route — {best_single_name}',
    output_name=f'{slugify(best_single_name)}_best_single_route_map.html',
)

current_routes = []
if current_choice['winner_type'] == 'single_truck':
    current_routes.append(
        {
            'truck_name': current_choice['winner_name'],
            'display_name': f"Recommended truck: {current_choice['winner_name']}",
            'route': current_choice['detail']['route'].copy(),
            'color': '#2c7fb8',
            'icon_color': '#2c7fb8',
        }
    )
else:
    current_routes.extend([
        {
            'truck_name': current_choice['truck_1'],
            'display_name': f"{current_choice['truck_1']} — Truck 1",
            'route': current_choice['detail_1']['route'].copy(),
            'color': '#e74c3c',
            'icon_color': '#e74c3c',
        },
        {
            'truck_name': current_choice['truck_2'],
            'display_name': f"{current_choice['truck_2']} — Truck 2",
            'route': current_choice['detail_2']['route'].copy(),
            'color': '#27ae60',
            'icon_color': '#27ae60',
        },
    ])

recommended_route_map, recommended_map_path = build_polished_route_map(
    current_routes,
    title=f"Recommended Route Map — {current_choice['winner_name']}",
    output_name='recommended_route_map.html',
)

print('Route maps saved:')
print(f'- {best_single_map_path}')
print(f'- {recommended_map_path}')

display(HTML('<b>Download route maps:</b>'))
display(FileLink(str(best_single_map_path)))
display(FileLink(str(recommended_map_path)))

print('Best single-truck route map preview:')
display(best_single_route_map)
print('Recommended route map preview:')
display(recommended_route_map)

Route maps saved:
- /Users/macbookair/Desktop/SMU/ITOM 3355/Project One - Group Challenge/Cuy-Delivery/route_maps/condor_best_single_route_map.html
- /Users/macbookair/Desktop/SMU/ITOM 3355/Project One - Group Challenge/Cuy-Delivery/route_maps/recommended_route_map.html


/Users/macbookair/Desktop/SMU/ITOM 3355/Project One - Group Challenge/Cuy-Delivery/route_maps/condor_best_single_route_map.html

/Users/macbookair/Desktop/SMU/ITOM 3355/Project One - Group Challenge/Cuy-Delivery/route_maps/recommended_route_map.html

Best single-truck route map preview:


Recommended route map preview:


## Submission Checklist

Before submitting, confirm:

- [ ] Notebook runs top-to-bottom without errors
- [ ] `DAILY_SUPPLY_CAP` is set as a single variable (never hardcoded downstream)
- [ ] HQ address is geocoded; all routes start and end at HQ
- [ ] All 5+ assert statements pass
- [ ] At least 3 scenarios evaluated at `DAILY_SUPPLY_CAP = 165` (shown in comparison table)
- [ ] At least one additional supply level tested (e.g., 100 or 120 entrees)
- [ ] Final P&L table for your recommended configuration
- [ ] `Our Recommendation` markdown cell is complete with specific numbers
- [ ] Notebook is committed and pushed to your group's GitHub repo
- [ ] GitHub repo link submitted on Canvas before start of X11D2

**File name:** `X11_Project1_[GroupName].ipynb`  
**Canvas assignment:** Project One — Group Challenge  
**Due:** X11D2 (Thursday), start of class  

> ⚡ On X11D2 your instructor will announce the actual supply available for that day.
> Be ready to update `DAILY_SUPPLY_CAP`, re-run, and present your result within minutes.
